# Bài 04 — Định thức và ý nghĩa hình học với NumPy

Notebook này giúp bạn nhìn determinant từ ba góc: công thức đại số, hệ số co giãn thể tích có hướng và log-volume trong mô hình xác suất.

## Mục tiêu

Sau khi hoàn thành, bạn có thể:

- tính determinant `2×2`, `3×3` và kiểm chứng bằng NumPy;
- đo diện tích có hướng của hình sau phép biến đổi;
- suy luận determinant qua biến đổi hàng và các tính chất ma trận;
- phân biệt determinant magnitude với condition number;
- dùng `slogdet` cho log-volume ổn định;
- tính Gaussian log-density và xây Gaussian classifier chỉ bằng NumPy.

> Trước khi chạy mỗi cell, hãy dự đoán dấu, magnitude và shape của kết quả.

## 0. Khởi tạo

In [1]:
import numpy as np

np.set_printoptions(precision=6, suppress=True)
rng = np.random.default_rng(42)

def show(name, value):
    array = np.asarray(value)
    print(f"{name}: shape={array.shape}, ndim={array.ndim}, dtype={array.dtype}")
    print(array, "\n")

def det2(matrix):
    matrix = np.asarray(matrix, dtype=float)
    if matrix.shape != (2, 2):
        raise ValueError("det2 chỉ nhận ma trận shape (2, 2)")
    a, b = matrix[0]
    c, d = matrix[1]
    return a * d - b * c

def signed_polygon_area(vertices):
    vertices = np.asarray(vertices, dtype=float)
    x = vertices[:, 0]
    y = vertices[:, 1]
    return 0.5 * (x @ np.roll(y, -1) - y @ np.roll(x, -1))

print("NumPy:", np.__version__)

NumPy: 2.4.3


## 1. Determinant `2×2`: công thức `ad - bc`

Ta tính tay trước rồi đối chiếu `np.linalg.det`. Kết quả floating point được kiểm tra bằng `allclose`.

In [2]:
A = np.array([[2.0, 1.0], [0.0, 3.0]])
det_A_manual = det2(A)
det_A_numpy = np.linalg.det(A)

show("A", A)
print("det2(A):", det_A_manual)
print("np.linalg.det(A):", det_A_numpy)
np.testing.assert_allclose(det_A_manual, 6.0)
np.testing.assert_allclose(det_A_manual, det_A_numpy)

A: shape=(2, 2), ndim=2, dtype=float64
[[2. 1.]
 [0. 3.]] 

det2(A): 6.0
np.linalg.det(A): 6.0


## 2. `|det(A)|` là hệ số co giãn diện tích

Các điểm được lưu theo hàng, nên phép biến đổi cột `z = A @ x` được viết thành `Z = X @ A.T`. Shoelace formula cho signed area của polygon.

In [3]:
unit_square = np.array([[0.0, 0.0], [1.0, 0.0], [1.0, 1.0], [0.0, 1.0]])
transformed_square = unit_square @ A.T
area_before = signed_polygon_area(unit_square)
area_after = signed_polygon_area(transformed_square)

show("unit_square", unit_square)
show("transformed_square", transformed_square)
print("signed area trước/sau:", area_before, area_after)
np.testing.assert_allclose(area_before, 1.0)
np.testing.assert_allclose(area_after, np.linalg.det(A) * area_before)
np.testing.assert_allclose(abs(area_after), abs(np.linalg.det(A)))

unit_square: shape=(4, 2), ndim=2, dtype=float64
[[0. 0.]
 [1. 0.]
 [1. 1.]
 [0. 1.]] 

transformed_square: shape=(4, 2), ndim=2, dtype=float64
[[0. 0.]
 [2. 0.]
 [3. 3.]
 [1. 3.]] 

signed area trước/sau: 1.0 6.0


### Dấu âm: diện tích giữ dương nhưng orientation bị đảo

Phản xạ qua trục tung có determinant `-1`. Signed area đổi dấu; diện tích hình học là trị tuyệt đối nên vẫn bằng 1.

In [4]:
reflection = np.diag([-1.0, 1.0])
reflected_square = unit_square @ reflection.T
signed_reflected_area = signed_polygon_area(reflected_square)

print("det(reflection):", np.linalg.det(reflection))
print("signed area:", signed_reflected_area)
np.testing.assert_allclose(signed_reflected_area, -1.0)
np.testing.assert_allclose(abs(signed_reflected_area), 1.0)

det(reflection): -1.0
signed area: -1.0


## 3. Determinant `3×3` bằng khai triển hàng đầu

Cofactor giúp học công thức với ma trận nhỏ. Không dùng cách đệ quy này cho ma trận lớn.

In [5]:
C = np.array([[1.0, 2.0, 0.0], [3.0, 1.0, 1.0], [2.0, 0.0, 1.0]])
a, b, c = C[0]
minor_00 = det2(C[1:, 1:])
minor_01 = det2(C[1:, [0, 2]])
minor_02 = det2(C[1:, :2])
det_C_manual = a * minor_00 - b * minor_01 + c * minor_02

show("C", C)
print("minors:", minor_00, minor_01, minor_02)
print("det(C):", det_C_manual)
np.testing.assert_allclose(det_C_manual, -1.0)
np.testing.assert_allclose(det_C_manual, np.linalg.det(C))

C: shape=(3, 3), ndim=2, dtype=float64
[[1. 2. 0.]
 [3. 1. 1.]
 [2. 0. 1.]] 

minors: 1.0 1.0 -2.0
det(C): -1.0


## 4. Biến đổi hàng: ba quy tắc khác nhau

Đổi hai hàng đổi dấu; scale một hàng scale determinant; cộng bội hàng khác không đổi determinant.

In [6]:
row_A = np.array([[2.0, 1.0], [3.0, 4.0]])
base_det = np.linalg.det(row_A)

row_swapped = row_A[[1, 0]]
row_scaled = row_A.copy()
row_scaled[0] *= 5.0
row_added = row_A.copy()
row_added[1] += 7.0 * row_added[0]

print("base / swapped / scaled / row-added:")
print(base_det, np.linalg.det(row_swapped), np.linalg.det(row_scaled), np.linalg.det(row_added))
np.testing.assert_allclose(np.linalg.det(row_swapped), -base_det)
np.testing.assert_allclose(np.linalg.det(row_scaled), 5.0 * base_det)
np.testing.assert_allclose(np.linalg.det(row_added), base_det)

base / swapped / scaled / row-added:
5.000000000000001 -5.000000000000001 25.000000000000007 5.000000000000002


### Scale một hàng khác scale toàn ma trận

Với ma trận `n×n`, `det(cA) = c^n det(A)` vì cả `n` hàng đều được scale.

In [7]:
scale = 3.0
n = C.shape[0]
np.testing.assert_allclose(np.linalg.det(scale * C), scale ** n * np.linalg.det(C))
print("det(3C):", np.linalg.det(scale * C), "= 3^3 det(C):", scale ** n * np.linalg.det(C))

det(3C): -27.0 = 3^3 det(C): -27.00000000000001


## 5. Tính chất của tích, chuyển vị và nghịch đảo

`det` nhân qua matrix product, giữ nguyên qua transpose và lấy nghịch đảo qua reciprocal.

In [8]:
prop_A = np.array([[2.0, 1.0], [1.0, 3.0]])
prop_B = np.array([[1.0, 2.0], [0.0, -1.0]])
det_prop_A = np.linalg.det(prop_A)
det_prop_B = np.linalg.det(prop_B)

np.testing.assert_allclose(np.linalg.det(prop_A @ prop_B), det_prop_A * det_prop_B)
np.testing.assert_allclose(np.linalg.det(prop_A.T), det_prop_A)
np.testing.assert_allclose(np.linalg.det(np.linalg.inv(prop_A)), 1.0 / det_prop_A)
np.testing.assert_allclose(np.linalg.det(np.linalg.matrix_power(prop_A, 3)), det_prop_A ** 3)

print("det(A), det(B), det(AB):", det_prop_A, det_prop_B, np.linalg.det(prop_A @ prop_B))

det(A), det(B), det(AB): 5.000000000000001 -1.0 -5.000000000000001


### Determinant không phân phối qua phép cộng

Một phản ví dụ đủ để bác bỏ công thức tổng quát `det(A+B) = det(A)+det(B)`.

In [9]:
det_sum = np.linalg.det(prop_A + prop_B)
sum_dets = det_prop_A + det_prop_B
print("det(A+B):", det_sum, "det(A)+det(B):", sum_dets)
assert not np.isclose(det_sum, sum_dets)

det(A+B): 3.0000000000000004 det(A)+det(B): 4.000000000000001


## 6. Ma trận tam giác: tích đường chéo

Các phần tử ngoài đường chéo không ảnh hưởng determinant của ma trận tam giác.

In [10]:
upper = np.array([[2.0, 4.0, -1.0], [0.0, 3.0, 5.0], [0.0, 0.0, -2.0]])
diagonal_product = np.prod(np.diag(upper))

show("upper", upper)
print("tích đường chéo:", diagonal_product)
np.testing.assert_allclose(diagonal_product, -12.0)
np.testing.assert_allclose(np.linalg.det(upper), diagonal_product)

upper: shape=(3, 3), ndim=2, dtype=float64
[[ 2.  4. -1.]
 [ 0.  3.  5.]
 [ 0.  0. -2.]] 

tích đường chéo: -12.0


## 7. Determinant bằng 0 và mất chiều

Hai cột phụ thuộc làm hình vuông bị ép thành đoạn thẳng. Rank giảm và condition number trở nên cực lớn hoặc vô hạn.

In [11]:
singular = np.array([[2.0, 4.0], [1.0, 2.0]])
collapsed_square = unit_square @ singular.T

show("collapsed_square", collapsed_square)
print("det:", np.linalg.det(singular))
print("rank:", np.linalg.matrix_rank(singular))
print("condition number:", np.linalg.cond(singular))
np.testing.assert_allclose(np.linalg.det(singular), 0.0, atol=1e-12)
np.testing.assert_allclose(signed_polygon_area(collapsed_square), 0.0, atol=1e-12)
assert np.linalg.matrix_rank(singular) == 1

collapsed_square: shape=(4, 2), ndim=2, dtype=float64
[[0. 0.]
 [2. 1.]
 [6. 3.]
 [4. 2.]] 

det: 0.0
rank: 1
condition number: inf


## 8. Determinant magnitude không phải condition number

Ma trận có determinant rất nhỏ vẫn có thể condition tốt; determinant bằng 1 vẫn có thể condition cực xấu.

In [12]:
small_but_balanced = np.diag([1e-6, 1e-6])
unit_det_but_unbalanced = np.diag([1e6, 1e-6])

for name, matrix in [("small/balanced", small_but_balanced), ("unit-det/unbalanced", unit_det_but_unbalanced)]:
    print(name, "det=", np.linalg.det(matrix), "cond=", np.linalg.cond(matrix))

np.testing.assert_allclose(np.linalg.cond(small_but_balanced), 1.0)
np.testing.assert_allclose(np.linalg.det(unit_det_but_unbalanced), 1.0)
assert np.linalg.cond(unit_det_but_unbalanced) > 1e11

small/balanced det= 1.000000000000001e-12 cond= 1.0
unit-det/unbalanced det= 1.0 cond= 1000000000000.0


## 9. `slogdet`: log-volume không overflow

Determinant của ma trận chéo dưới đây là `100^200 = 10^400`, vượt phạm vi `float64`. Log-determinant vẫn hữu hạn.

In [13]:
huge_diagonal = np.eye(200) * 100.0
with np.errstate(over="ignore"):
    direct_det = np.linalg.det(huge_diagonal)
sign, logabsdet = np.linalg.slogdet(huge_diagonal)
expected_logabsdet = 200 * np.log(100.0)

print("direct det:", direct_det)
print("sign / log|det|:", sign, logabsdet)
assert np.isinf(direct_det)
assert sign == 1.0
np.testing.assert_allclose(logabsdet, expected_logabsdet)

direct det: inf
sign / log|det|: 1.0 921.0340371976157


## 10. Determinant theo batch

`np.linalg.det` hoạt động trên hai trục cuối và giữ các batch axis đứng trước.

In [14]:
batch_matrices = np.stack([np.eye(2), A, reflection, singular])
batch_dets = np.linalg.det(batch_matrices)
batch_signs, batch_logabs = np.linalg.slogdet(batch_matrices)

show("batch_matrices", batch_matrices)
print("determinants:", batch_dets)
print("signs:", batch_signs)
assert batch_dets.shape == (4,)
np.testing.assert_allclose(batch_dets, [1.0, 6.0, -1.0, 0.0], atol=1e-12)
assert np.isneginf(batch_logabs[-1])

batch_matrices: shape=(4, 2, 2), ndim=3, dtype=float64
[[[ 1.  0.]
  [ 0.  1.]]

 [[ 2.  1.]
  [ 0.  3.]]

 [[-1.  0.]
  [ 0.  1.]]

 [[ 2.  4.]
  [ 1.  2.]]] 

determinants: [ 1.  6. -1.  0.]
signs: [ 1.  1. -1.  0.]


## 11. Gaussian log-density bằng `slogdet + solve`

Log-determinant chuẩn hóa volume; `solve` tính Mahalanobis term mà không tạo nghịch đảo covariance.

In [15]:
def gaussian_log_density(points, mean, covariance):
    points = np.asarray(points, dtype=float)
    mean = np.asarray(mean, dtype=float)
    covariance = np.asarray(covariance, dtype=float)
    if points.ndim != 2 or mean.shape != (points.shape[1],):
        raise ValueError("Shape của points/mean không tương thích")
    if covariance.shape != (mean.size, mean.size):
        raise ValueError("Covariance phải có shape (d, d)")
    sign, logdet = np.linalg.slogdet(covariance)
    if sign <= 0:
        raise ValueError("Covariance phải có determinant dương")
    centered = points - mean
    solved = np.linalg.solve(covariance, centered.T).T
    mahalanobis_sq = np.sum(centered * solved, axis=1)
    return -0.5 * (mean.size * np.log(2.0 * np.pi) + logdet + mahalanobis_sq)

gaussian_mean = np.array([0.0, 1.0])
gaussian_cov = np.array([[2.0, 0.6], [0.6, 1.0]])
gaussian_points = np.array([[0.0, 1.0], [1.0, 1.5], [3.0, -1.0]])
log_densities = gaussian_log_density(gaussian_points, gaussian_mean, gaussian_cov)

print("log densities:", log_densities)
assert log_densities.shape == (3,)
assert np.argmax(log_densities) == 0

log densities: [-2.085225 -2.359615 -9.463274]


## 12. Covariance suy biến và diagonal regularization

Feature thứ hai là bội của feature thứ nhất làm covariance determinant bằng 0. Cộng `λI` tạo determinant dương và cho phép tính density.

In [16]:
dependent_data = np.array([[1.0, 2.0], [2.0, 4.0], [3.0, 6.0], [4.0, 8.0]])
centered_data = dependent_data - dependent_data.mean(axis=0)
singular_cov = centered_data.T @ centered_data / len(dependent_data)
regularization = 1e-2
regularized_cov = singular_cov + regularization * np.eye(2)

print("det covariance trước/sau:", np.linalg.det(singular_cov), np.linalg.det(regularized_cov))
print("cond covariance trước/sau:", np.linalg.cond(singular_cov), np.linalg.cond(regularized_cov))
np.testing.assert_allclose(np.linalg.det(singular_cov), 0.0, atol=1e-12)
assert np.linalg.slogdet(regularized_cov)[0] > 0
assert np.linalg.cond(regularized_cov) < np.linalg.cond(singular_cov)

det covariance trước/sau: 0.0 0.06259999999999932
cond covariance trước/sau: 1.5734929547254926e+16 625.9999999999634


## Bài tập

Có 7 bài tập tăng dần và một bài tổng hợp. Với mỗi bài:

1. dự đoán dấu, magnitude và shape;
2. bỏ dấu `#` và hoàn thành `TODO`;
3. chạy ô bài làm rồi ô kiểm tra;
4. chỉ mở đáp án sau khi đã thử.

Các checker chỉ nhắc biến còn thiếu nên notebook vẫn chạy từ đầu đến cuối khi chưa làm bài.

### Bài 1 — Công thức `2×2`

Tính `ex1_det` bằng phép toán scalar `ad - bc`, không gọi `det2` hoặc `np.linalg.det`.

In [17]:
ex1_A = np.array([[4.0, -2.0], [3.0, 5.0]])

# TODO
# ex1_det = ...

In [18]:
if "ex1_det" not in globals():
    print("⏳ Chưa làm Bài 1: hãy tạo ex1_det.")
else:
    assert np.asarray(ex1_det).shape == ()
    np.testing.assert_allclose(ex1_det, np.linalg.det(ex1_A))
    np.testing.assert_allclose(ex1_det, 26.0)
    print("✅ Bài 1 đúng")

⏳ Chưa làm Bài 1: hãy tạo ex1_det.


<details><summary>Đáp án Bài 1</summary>

```python
ex1_det = ex1_A[0, 0] * ex1_A[1, 1] - ex1_A[0, 1] * ex1_A[1, 0]
```
</details>

### Bài 2 — Area và orientation

Biến đổi `ex2_square` bằng `ex2_A`, lưu các đỉnh vào `ex2_transformed`, signed area vào `ex2_signed_area` và diện tích hình học vào `ex2_area`.

In [19]:
ex2_square = unit_square.copy()
ex2_A = np.array([[3.0, 0.0], [0.0, -2.0]])

# TODO
# ex2_transformed = ...
# ex2_signed_area = ...
# ex2_area = ...

In [20]:
ex2_names = ["ex2_transformed", "ex2_signed_area", "ex2_area"]
if not all(name in globals() for name in ex2_names):
    print("⏳ Chưa làm Bài 2: hãy tạo", ex2_names)
else:
    assert ex2_transformed.shape == (4, 2)
    np.testing.assert_allclose(ex2_transformed, ex2_square @ ex2_A.T)
    np.testing.assert_allclose(ex2_signed_area, -6.0)
    np.testing.assert_allclose(ex2_area, 6.0)
    print("✅ Bài 2 đúng")

⏳ Chưa làm Bài 2: hãy tạo ['ex2_transformed', 'ex2_signed_area', 'ex2_area']


<details><summary>Đáp án Bài 2</summary>

```python
ex2_transformed = ex2_square @ ex2_A.T
ex2_signed_area = signed_polygon_area(ex2_transformed)
ex2_area = abs(ex2_signed_area)
```
</details>

### Bài 3 — Theo dõi biến đổi hàng

Từ `ex3_A`: đổi hàng 0 và 2, nhân hàng 1 với `-3`, rồi cộng `5 × hàng 0` vào hàng 2. Tạo `ex3_B` và suy luận `ex3_expected_det` từ `ex3_det_A` mà không gọi `det` cho `ex3_B`.

In [21]:
ex3_A = np.array([[1.0, 2.0, 0.0], [0.0, 3.0, 1.0], [2.0, 1.0, 4.0]])
ex3_det_A = np.linalg.det(ex3_A)

# TODO
# ex3_B = ex3_A.copy()
# ...
# ex3_expected_det = ...

In [22]:
if "ex3_B" not in globals() or "ex3_expected_det" not in globals():
    print("⏳ Chưa làm Bài 3: hãy tạo ex3_B và ex3_expected_det.")
else:
    expected_matrix = ex3_A[[2, 1, 0]].copy()
    expected_matrix[1] *= -3.0
    expected_matrix[2] += 5.0 * expected_matrix[0]
    np.testing.assert_allclose(ex3_B, expected_matrix)
    np.testing.assert_allclose(ex3_expected_det, 3.0 * ex3_det_A)
    np.testing.assert_allclose(np.linalg.det(ex3_B), ex3_expected_det)
    print("✅ Bài 3 đúng")

⏳ Chưa làm Bài 3: hãy tạo ex3_B và ex3_expected_det.


<details><summary>Đáp án Bài 3</summary>

```python
ex3_B = ex3_A[[2, 1, 0]].copy()  # swap: nhân determinant với -1
ex3_B[1] *= -3.0                 # scale: nhân thêm -3
ex3_B[2] += 5.0 * ex3_B[0]      # row addition: không đổi
ex3_expected_det = 3.0 * ex3_det_A
```
Hai hệ số `-1` và `-3` nhân nhau thành `3`.
</details>

### Bài 4 — Determinant của tích và nghịch đảo

Không tính `det(ex4_A @ ex4_B)` trực tiếp. Tạo `ex4_det_product` từ hai determinant riêng và `ex4_det_inverse_product` cho nghịch đảo của tích.

In [23]:
ex4_A = np.array([[2.0, 1.0], [1.0, 1.0]])
ex4_B = np.array([[3.0, -1.0], [2.0, 2.0]])

# TODO
# ex4_det_A = ...
# ex4_det_B = ...
# ex4_det_product = ...
# ex4_det_inverse_product = ...

In [24]:
ex4_names = ["ex4_det_A", "ex4_det_B", "ex4_det_product", "ex4_det_inverse_product"]
if not all(name in globals() for name in ex4_names):
    print("⏳ Chưa làm Bài 4: hãy tạo", ex4_names)
else:
    np.testing.assert_allclose(ex4_det_product, np.linalg.det(ex4_A @ ex4_B))
    np.testing.assert_allclose(ex4_det_inverse_product, np.linalg.det(np.linalg.inv(ex4_A @ ex4_B)))
    np.testing.assert_allclose(ex4_det_product * ex4_det_inverse_product, 1.0)
    print("✅ Bài 4 đúng")

⏳ Chưa làm Bài 4: hãy tạo ['ex4_det_A', 'ex4_det_B', 'ex4_det_product', 'ex4_det_inverse_product']


<details><summary>Đáp án Bài 4</summary>

```python
ex4_det_A = np.linalg.det(ex4_A)
ex4_det_B = np.linalg.det(ex4_B)
ex4_det_product = ex4_det_A * ex4_det_B
ex4_det_inverse_product = 1.0 / ex4_det_product
```
</details>

### Bài 5 — Batch ma trận tam giác

Tạo `ex5_dets` bằng tích đường chéo trên trục cuối, không dùng `np.linalg.det`. Kết quả phải có một determinant cho mỗi ma trận.

In [25]:
ex5_batch = np.array([
    [[2.0, 1.0, 3.0], [0.0, 4.0, -1.0], [0.0, 0.0, 5.0]],
    [[-1.0, 2.0, 0.0], [0.0, 3.0, 4.0], [0.0, 0.0, 2.0]],
    [[1.0, 5.0, 2.0], [0.0, 0.0, 1.0], [0.0, 0.0, 7.0]],
])

# TODO
# ex5_diagonals = ...
# ex5_dets = ...

In [26]:
if "ex5_diagonals" not in globals() or "ex5_dets" not in globals():
    print("⏳ Chưa làm Bài 5: hãy tạo ex5_diagonals và ex5_dets.")
else:
    assert ex5_diagonals.shape == (3, 3)
    assert ex5_dets.shape == (3,)
    np.testing.assert_allclose(ex5_dets, np.linalg.det(ex5_batch))
    np.testing.assert_allclose(ex5_dets, [40.0, -6.0, 0.0])
    print("✅ Bài 5 đúng")

⏳ Chưa làm Bài 5: hãy tạo ex5_diagonals và ex5_dets.


<details><summary>Đáp án Bài 5</summary>

```python
ex5_diagonals = np.diagonal(ex5_batch, axis1=-2, axis2=-1)
ex5_dets = np.prod(ex5_diagonals, axis=1)
```
</details>

### Bài 6 — Log-determinant không tạo determinant

Ma trận chéo `ex6_A` có phần tử dương. Tạo `ex6_sign` và `ex6_logabsdet` bằng `slogdet`, rồi kiểm chứng bằng tổng log đường chéo vào `ex6_manual_logabsdet`.

In [27]:
ex6_diagonal = np.geomspace(1e-20, 1e20, 100)
ex6_A = np.diag(ex6_diagonal)

# TODO
# ex6_sign, ex6_logabsdet = ...
# ex6_manual_logabsdet = ...

In [28]:
ex6_names = ["ex6_sign", "ex6_logabsdet", "ex6_manual_logabsdet"]
if not all(name in globals() for name in ex6_names):
    print("⏳ Chưa làm Bài 6: hãy tạo", ex6_names)
else:
    assert ex6_sign == 1.0
    np.testing.assert_allclose(ex6_logabsdet, ex6_manual_logabsdet, atol=1e-10)
    np.testing.assert_allclose(ex6_logabsdet, 0.0, atol=1e-10)
    print("✅ Bài 6 đúng")

⏳ Chưa làm Bài 6: hãy tạo ['ex6_sign', 'ex6_logabsdet', 'ex6_manual_logabsdet']


<details><summary>Đáp án Bài 6</summary>

```python
ex6_sign, ex6_logabsdet = np.linalg.slogdet(ex6_A)
ex6_manual_logabsdet = np.log(ex6_diagonal).sum()
```
Dãy geometric đối xứng trong log-space nên tổng log xấp xỉ 0.
</details>

### Bài 7 — Gaussian log-density vector hóa

Không gọi hàm mẫu `gaussian_log_density`. Tự tạo `ex7_log_density` cho mọi điểm bằng `slogdet`, `solve` và các phép NumPy vector hóa. Không dùng `inv`.

In [29]:
ex7_points = np.array([[0.0, 0.0], [1.0, -1.0], [2.0, 2.0], [-1.0, 0.5]])
ex7_mean = np.array([0.5, -0.5])
ex7_cov = np.array([[1.5, 0.3], [0.3, 0.8]])

# TODO
# ex7_sign, ex7_logdet = ...
# ex7_centered = ...
# ex7_solved = ...
# ex7_mahalanobis_sq = ...
# ex7_log_density = ...

In [30]:
ex7_names = ["ex7_sign", "ex7_logdet", "ex7_centered", "ex7_solved", "ex7_mahalanobis_sq", "ex7_log_density"]
if not all(name in globals() for name in ex7_names):
    print("⏳ Chưa làm Bài 7: hãy tạo", ex7_names)
else:
    assert ex7_sign > 0
    assert ex7_centered.shape == (4, 2)
    assert ex7_solved.shape == (4, 2)
    assert ex7_mahalanobis_sq.shape == (4,)
    assert ex7_log_density.shape == (4,)
    np.testing.assert_allclose(ex7_log_density, gaussian_log_density(ex7_points, ex7_mean, ex7_cov))
    print("✅ Bài 7 đúng")

⏳ Chưa làm Bài 7: hãy tạo ['ex7_sign', 'ex7_logdet', 'ex7_centered', 'ex7_solved', 'ex7_mahalanobis_sq', 'ex7_log_density']


<details><summary>Đáp án Bài 7</summary>

```python
ex7_sign, ex7_logdet = np.linalg.slogdet(ex7_cov)
ex7_centered = ex7_points - ex7_mean
ex7_solved = np.linalg.solve(ex7_cov, ex7_centered.T).T
ex7_mahalanobis_sq = np.sum(ex7_centered * ex7_solved, axis=1)
ex7_log_density = -0.5 * (
    ex7_mean.size * np.log(2.0 * np.pi)
    + ex7_logdet
    + ex7_mahalanobis_sq
)
```
</details>

## Bài tổng hợp — Gaussian classifier với covariance riêng cho từng lớp

Hai lớp có dữ liệu huấn luyện 2D. Hãy:

1. tính mean mỗi lớp;
2. tính maximum-likelihood covariance bằng `centered.T @ centered / N`;
3. thêm `cap_lambda * I` vào mỗi covariance;
4. viết `cap_log_density` dùng `slogdet + solve`;
5. tạo `cap_scores(N_test, 2)` và `cap_predictions(N_test,)`;
6. lưu signs/log-determinants và condition numbers để chẩn đoán.

Không dùng `det` hay `inv`. Giả sử hai lớp có prior bằng nhau nên không cần cộng log-prior.

In [31]:
cap_class0 = np.array([[-1.0, 0.0], [-0.8, 0.2], [-1.2, -0.1], [-0.9, 0.1], [-1.1, -0.2]])
cap_class1 = np.array([[1.0, 1.0], [1.2, 0.8], [0.8, 1.1], [1.1, 1.2], [0.9, 0.9]])
cap_test = np.array([[-1.0, 0.1], [1.0, 1.0], [0.0, 0.5], [-0.4, 0.0]])
cap_lambda = 0.05

# TODO 1: means
# cap_mean0 = ...
# cap_mean1 = ...

# TODO 2: covariance MLE
# cap_centered0 = ...
# cap_centered1 = ...
# cap_cov0_raw = ...
# cap_cov1_raw = ...

# TODO 3: diagonal regularization
# cap_cov0 = ...
# cap_cov1 = ...

# TODO 4: hàm log-density
# def cap_log_density(points, mean, covariance):
#     ...

# TODO 5: scores và predictions
# cap_score0 = ...
# cap_score1 = ...
# cap_scores = ...
# cap_predictions = ...

# TODO 6: chẩn đoán
# cap_sign0, cap_logdet0 = ...
# cap_sign1, cap_logdet1 = ...
# cap_condition0 = ...
# cap_condition1 = ...

In [32]:
cap_names = [
    "cap_mean0", "cap_mean1", "cap_centered0", "cap_centered1",
    "cap_cov0_raw", "cap_cov1_raw", "cap_cov0", "cap_cov1",
    "cap_log_density", "cap_score0", "cap_score1", "cap_scores", "cap_predictions",
    "cap_sign0", "cap_logdet0", "cap_sign1", "cap_logdet1",
    "cap_condition0", "cap_condition1",
]
if not all(name in globals() for name in cap_names):
    missing = [name for name in cap_names if name not in globals()]
    print("⏳ Bài tổng hợp chưa đủ. Còn thiếu:", missing)
else:
    assert cap_mean0.shape == (2,) and cap_mean1.shape == (2,)
    assert cap_cov0.shape == (2, 2) and cap_cov1.shape == (2, 2)
    assert cap_score0.shape == (4,) and cap_score1.shape == (4,)
    assert cap_scores.shape == (4, 2) and cap_predictions.shape == (4,)
    np.testing.assert_allclose(cap_cov0, cap_cov0.T)
    np.testing.assert_allclose(cap_cov1, cap_cov1.T)
    np.testing.assert_allclose(cap_score0, gaussian_log_density(cap_test, cap_mean0, cap_cov0))
    np.testing.assert_allclose(cap_score1, gaussian_log_density(cap_test, cap_mean1, cap_cov1))
    np.testing.assert_array_equal(cap_predictions, np.argmax(cap_scores, axis=1))
    np.testing.assert_array_equal(cap_predictions[:2], [0, 1])
    assert cap_sign0 > 0 and cap_sign1 > 0
    assert np.isfinite(cap_scores).all()
    assert cap_condition0 < np.linalg.cond(cap_cov0_raw)
    assert cap_condition1 < np.linalg.cond(cap_cov1_raw)
    print("✅ Bài tổng hợp đúng")
    print("means:", cap_mean0, cap_mean1)
    print("sign/logdet lớp 0:", cap_sign0, cap_logdet0)
    print("sign/logdet lớp 1:", cap_sign1, cap_logdet1)
    print("condition numbers:", cap_condition0, cap_condition1)
    print("scores:\n", cap_scores)
    print("predictions:", cap_predictions)

⏳ Bài tổng hợp chưa đủ. Còn thiếu: ['cap_mean0', 'cap_mean1', 'cap_centered0', 'cap_centered1', 'cap_cov0_raw', 'cap_cov1_raw', 'cap_cov0', 'cap_cov1', 'cap_log_density', 'cap_score0', 'cap_score1', 'cap_scores', 'cap_predictions', 'cap_sign0', 'cap_logdet0', 'cap_sign1', 'cap_logdet1', 'cap_condition0', 'cap_condition1']


<details><summary>Đáp án bài tổng hợp</summary>

```python
cap_mean0 = cap_class0.mean(axis=0)
cap_mean1 = cap_class1.mean(axis=0)

cap_centered0 = cap_class0 - cap_mean0
cap_centered1 = cap_class1 - cap_mean1
cap_cov0_raw = cap_centered0.T @ cap_centered0 / len(cap_class0)
cap_cov1_raw = cap_centered1.T @ cap_centered1 / len(cap_class1)

cap_cov0 = cap_cov0_raw + cap_lambda * np.eye(2)
cap_cov1 = cap_cov1_raw + cap_lambda * np.eye(2)

def cap_log_density(points, mean, covariance):
    sign, logdet = np.linalg.slogdet(covariance)
    if sign <= 0:
        raise ValueError("Covariance phải có determinant dương")
    centered = points - mean
    solved = np.linalg.solve(covariance, centered.T).T
    mahalanobis_sq = np.sum(centered * solved, axis=1)
    return -0.5 * (
        mean.size * np.log(2.0 * np.pi) + logdet + mahalanobis_sq
    )

cap_score0 = cap_log_density(cap_test, cap_mean0, cap_cov0)
cap_score1 = cap_log_density(cap_test, cap_mean1, cap_cov1)
cap_scores = np.column_stack([cap_score0, cap_score1])
cap_predictions = np.argmax(cap_scores, axis=1)

cap_sign0, cap_logdet0 = np.linalg.slogdet(cap_cov0)
cap_sign1, cap_logdet1 = np.linalg.slogdet(cap_cov1)
cap_condition0 = np.linalg.cond(cap_cov0)
cap_condition1 = np.linalg.cond(cap_cov1)
```

Regularization làm covariance dễ giải hơn. Log-determinant và Mahalanobis distance cùng quyết định score của mỗi lớp.
</details>

## Tự kiểm tra nhanh

Không chạy code, hãy trả lời:

1. `det(A) = -4` nói gì về area scale và orientation?
2. Vì sao hình vuông đơn vị biến thành hình bình hành sinh bởi hai cột của `A`?
3. Đổi hai hàng, scale một hàng và cộng bội hàng ảnh hưởng determinant thế nào?
4. Với `A` là `4×4`, `det(3A)` liên hệ với `det(A)` ra sao?
5. Vì sao determinant của ma trận tam giác là tích đường chéo?
6. `det(A) = 1` có bảo đảm condition number nhỏ không?
7. `slogdet` giải quyết vấn đề gì của `log(abs(det(A)))`?
8. Nếu `matrices.shape == (16, 3, 3)`, `det(matrices)` có shape gì?
9. Log-determinant và Mahalanobis term đóng vai trò gì trong Gaussian log-density?
10. Vì sao thêm `λI` có thể làm covariance suy biến trở nên dùng được?
11. Trong change of variables, vì sao density phải giảm khi volume tăng?

## Tổng kết

- `det(A)` là scalar của ma trận vuông; `|det(A)|` đo volume scale và dấu đo orientation.
- Determinant 0 nghĩa là phép biến đổi làm mất chiều và không khả nghịch.
- Row swap đổi dấu, row scaling scale determinant, row addition giữ determinant.
- `det(AB) = det(A)det(B)`; transpose giữ determinant; inverse lấy reciprocal.
- Ma trận tam giác có determinant bằng tích đường chéo.
- Magnitude của determinant không thay thế condition number, rank hay singular values.
- `slogdet` giữ sign và log-volume ổn định cho bài toán xác suất nhiều chiều.
- Gaussian log-density kết hợp log-determinant với Mahalanobis distance và nên dùng `solve`, không dùng `inv`.
- Jacobian determinant điều chỉnh density khi phép đổi biến làm co giãn không gian.

Trước khi sang Bài 05, hãy tự tạo hai ma trận có cùng determinant nhưng condition number rất khác nhau và giải thích vì sao điều đó có thể xảy ra.